In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df_fake=pd.read_csv('politifact_fake.csv')
df_real=pd.read_csv('politifact_real.csv')

In [ ]:


# -----------------------------
# Prepare FAKE dataset
# -----------------------------
df_fake = df_fake[['title']].copy()      # keep only title
df_fake.rename(columns={'title': 'article'}, inplace=True)
df_fake['label'] = 0                     # 0 = fake

# -----------------------------
# Prepare REAL dataset
# -----------------------------
df_real = df_real[['title']].copy()      # keep only title
df_real.rename(columns={'title': 'article'}, inplace=True)
df_real['label'] = 1                     # 1 = real

# -----------------------------
# Combine both datasets
# -----------------------------
df = pd.concat([df_fake, df_real], ignore_index=True)

# -----------------------------
# Shuffle rows
# -----------------------------
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# -----------------------------
# Remove missing values
# -----------------------------
df.dropna(subset=['article'], inplace=True)

# -----------------------------
# Preview final dataset
# -----------------------------
print(df.head())
print("\nClass Distribution:")
print(df['label'].value_counts())

print("\nFinal Shape:", df.shape)

In [ ]:
df.to_csv("final_fake_real_dataset.csv", index=False)

print("Dataset saved successfully.")

In [ ]:
# Load dataset -> Split -> Save train/test sets

import pandas as pd
from sklearn.model_selection import train_test_split

# ------------------------------------------------
# Load your saved dataset
# ------------------------------------------------
df = pd.read_csv("final_fake_real_dataset.csv")

# ------------------------------------------------
# Features and target
# ------------------------------------------------
X = df["article"]
y = df["label"]

# ------------------------------------------------
# 80 / 20 Split
# ------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ------------------------------------------------
# Create train dataframe
# ------------------------------------------------
train_df = pd.DataFrame({
    "article": X_train.values,
    "label": y_train.values
})

# ------------------------------------------------
# Create test dataframe
# ------------------------------------------------
test_df = pd.DataFrame({
    "article": X_test.values,
    "label": y_test.values
})

# ------------------------------------------------
# Save files
# ------------------------------------------------
train_df.to_csv("train_dataset.csv", index=False)
test_df.to_csv("test_dataset.csv", index=False)

print("Files saved successfully.")
print("Train Shape:", train_df.shape)
print("Test Shape :", test_df.shape)

In [ ]:
df.shape[0]

In [ ]:
# Count how many REAL and FAKE rows are in dataset

print(df["label"].value_counts())

In [ ]:
!uv pip uninstall pycaret

In [ ]:
!pip uninstall -y numpy scikit-learn pandas
!pip install numpy==1.26.4 pandas==2.2.2 scikit-learn==1.4.2

In [ ]:
# PyCaret-like automatic model comparison for text classification
# Works in Google Colab / Python 3.12

import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# Metrics
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

# --------------------------------------------------
# df should contain:
# article = text column
# label   = target column (0=fake,1=real)
# --------------------------------------------------

X = df["article"]
y = df["label"]

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# --------------------------------------------------
# Models similar to PyCaret compare_models()
# --------------------------------------------------

models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "KNN": KNeighborsClassifier()
}

results = []

# --------------------------------------------------
# Train + Evaluate all models
# --------------------------------------------------

for name, model in models.items():

    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", max_features=30000)),
        ("model", model)
    ])

    # Fit
    pipe.fit(X_train, y_train)

    # Predict
    pred = pipe.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    prec = precision_score(y_test, pred)
    rec = recall_score(y_test, pred)

    # Cross Validation Score
    cv = cross_val_score(pipe, X, y, cv=3, scoring="accuracy").mean()

    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "F1 Score": round(f1, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "CV Score": round(cv, 4)
    })

# --------------------------------------------------
# Show Ranking Table
# --------------------------------------------------

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

print(results_df)

# --------------------------------------------------
# Best Model
# --------------------------------------------------

best_name = results_df.iloc[0]["Model"]
print("\nBest Model:", best_name)

In [ ]:
# Save model comparison results to CSV

results_df.to_csv("model_comparison_results_ML.csv", index=False)
results_df.to_excel("model_comparison_results_ML.xlsx", index=False)
results_df.to_markdown("model_comparison_results_ML.md", index=False)

print("Results saved successfully.")

In [ ]:
# ==========================================
# Test ALL models on random articles
# taken from your TEST SET
# ==========================================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# ---------------------------------------------------
# Prepare data
# ---------------------------------------------------

X = df["article"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ---------------------------------------------------
# Models
# ---------------------------------------------------

models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "KNN": KNeighborsClassifier()
}

# ---------------------------------------------------
# Select random test articles
# ---------------------------------------------------

sample_test = pd.DataFrame({
    "article": X_test,
    "true_label": y_test
}).sample(5, random_state=42).reset_index(drop=True)

# ---------------------------------------------------
# Predict each sample using all models
# ---------------------------------------------------

for i in range(len(sample_test)):

    article = sample_test.loc[i, "article"]
    true_label = sample_test.loc[i, "true_label"]

    print("="*80)
    print(f"TEST ARTICLE {i+1}")
    print("="*80)
    print(article[:500])   # first 500 chars
    print("\nActual Label:", "REAL" if true_label == 1 else "FAKE")
    print("-"*80)

    for name, model in models.items():

        pipe = Pipeline([
            ("tfidf", TfidfVectorizer(stop_words="english", max_features=30000)),
            ("model", model)
        ])

        # Train on training set only
        pipe.fit(X_train, y_train)

        # Predict this test article
        pred = pipe.predict([article])[0]

        print(f"{name:22s} --> {'REAL' if pred==1 else 'FAKE'}")

    print("\n")

In [ ]:
# ===============================================
# Test REAL-TIME News Article on ALL Trained Models
# ===============================================

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

# ------------------------------------------------
# Paste any real-time news article / headline here
# ------------------------------------------------

news_text = """
BREAKING: he company reported quarterly revenue growth of 8%, according to its earnings release.
"""

# ------------------------------------------------
# Same models as before
# ------------------------------------------------


models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "KNN": KNeighborsClassifier()
}

# ------------------------------------------------
# Train each model on FULL dataset then predict
# ------------------------------------------------

results = []

X = df["article"]
y = df["label"]

for name, model in models.items():

    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", max_features=30000)),
        ("model", model)
    ])

    # Train on full dataset
    pipe.fit(X, y)

    # Predict real-time article
    pred = pipe.predict([news_text])[0]

    # Probability if available
    confidence = "N/A"

    try:
        prob = pipe.predict_proba([news_text])[0]
        confidence = round(max(prob) * 100, 2)
    except:
        pass

    label = "REAL" if pred == 1 else "FAKE"

    results.append({
        "Model": name,
        "Prediction": label,
        "Confidence %": confidence
    })

# ------------------------------------------------
# Show Results
# ------------------------------------------------

results_df = pd.DataFrame(results)
print(results_df)

### **Testing on the real time custom dataset**

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

# --------------------------------------------------
# 1) Train data from your original dataset
# --------------------------------------------------
# df = your original dataframe
X = df["article"].fillna("")
y = df["label"]

# --------------------------------------------------
# 2) New external test dataset
# --------------------------------------------------
new_test_df = pd.read_csv("new_test_dataset.csv")

# Handle possible typo in column name
if "article" in new_test_df.columns:
    test_text_col = "article"
elif "atricle" in new_test_df.columns:
    test_text_col = "atricle"
else:
    raise ValueError("Text column not found in new test dataset. Expected 'article' or 'atricle'.")

X_new_test = new_test_df[test_text_col].fillna("")
y_new_test = new_test_df["label"]

# --------------------------------------------------
# 3) Models
# --------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "KNN": KNeighborsClassifier()
}

results = []
trained_models = {}

# --------------------------------------------------
# 4) Train on original data
# --------------------------------------------------
for name, model in models.items():

    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", max_features=30000)),
        ("model", model)
    ])

    pipe.fit(X, y)
    trained_models[name] = pipe

    # Evaluate on the new external test dataset
    pred = pipe.predict(X_new_test)

    acc = accuracy_score(y_new_test, pred)
    f1 = f1_score(y_new_test, pred, zero_division=0)
    prec = precision_score(y_new_test, pred, zero_division=0)
    rec = recall_score(y_new_test, pred, zero_division=0)

    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "F1 Score": round(f1, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4)
    })

    print("=" * 80)
    print(f"Model: {name}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_new_test, pred))
    print("\nClassification Report:")
    print(classification_report(y_new_test, pred, zero_division=0))

# --------------------------------------------------
# 5) Compare all models on new test dataset
# --------------------------------------------------
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("MODEL COMPARISON ON NEW TEST DATASET")
print(results_df)

best_name = results_df.iloc[0]["Model"]
print("\nBest Model on new test dataset:", best_name)